[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter9/graphrag-create.ipynb)

# Chapter 9: Microsoft GraphRAG for Movie Scripts

This notebook demonstrates using Microsoft's GraphRAG package to automatically extract knowledge graphs from movie scripts using LLMs.

**What you'll learn:**
- Configure and run Microsoft GraphRAG's indexing pipeline
- Automatically extract entities and relationships from unstructured text using LLMs
- Build hierarchical community structures for global understanding
- Estimate processing costs before running on large datasets

**Prerequisites:** `pip install graphrag tiktoken datasets` and set `OPENAI_API_KEY`.

## Step 1: Installation and Setup

We install Microsoft's GraphRAG package along with tiktoken for token counting and the datasets library for loading movie scripts.

In [9]:
# Install Microsoft GraphRAG
!pip install --quiet graphrag tiktoken datasets

/Users/ofer/miniconda3/envs/py312/lib/python3.12/site-packages/lancedb/__init__.py:241: UserWarning: lance is not fork-safe. If you are using multiprocessing, use spawn instead.
  warnings.warn(



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [10]:
import os
import pandas as pd
import asyncio
from pathlib import Path
from datasets import load_dataset

# GraphRAG imports - Updated for version 2.7.0
from graphrag.query.indexer_adapters import (
    read_indexer_entities,
    read_indexer_reports,
    read_indexer_covariates,
    read_indexer_relationships,
    read_indexer_text_units,
    read_indexer_communities,
)
from graphrag.query.factory import (
    get_global_search_engine,
    get_local_search_engine,
)
from graphrag.config.load_config import load_config
from graphrag.vector_stores.lancedb import LanceDBVectorStore

We set up workspace directories, API keys, and model choices — GPT-4.1-mini for entity extraction and text-embedding-3-small for vector search.

## Step 2: Configuration

We define the workspace paths, select the OpenAI models for extraction and embedding, and write the GraphRAG `settings.yaml` that controls chunking, entity extraction, and community report generation.

In [11]:
# Configuration
GRAPHRAG_DIR = Path("./graphrag_workspace")
INPUT_DIR = GRAPHRAG_DIR / "input"
OUTPUT_DIR = GRAPHRAG_DIR / "output"

# Create directories
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# OpenAI Configuration
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Please set OPENAI_API_KEY environment variable")

# Model configuration
LLM_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

print(f"GraphRAG workspace: {GRAPHRAG_DIR}")
print(f"Using LLM model: {LLM_MODEL}")
print(f"Using embedding model: {EMBEDDING_MODEL}")

GraphRAG workspace: graphrag_workspace
Using LLM model: gpt-4.1-mini
Using embedding model: text-embedding-3-small


## Step 3: Load Movie Scripts

We'll use 20 diverse movies from the dataset to balance cost and coverage. This provides enough data to build a meaningful knowledge graph while keeping costs manageable (~$7 vs ~$623 for all 1,800 movies).

In [13]:
# Load MovieSum dataset
print("Loading MovieSum dataset...")
moviesum_dataset = load_dataset("rohitsaxena/MovieSum")
moviesum_df = pd.DataFrame(moviesum_dataset['train'])

print(f"Total movies available: {len(moviesum_df)}")

# Select 20 diverse movies using stratified sampling
# This ensures we get a representative sample across the dataset
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Sample 20 movies evenly distributed across the dataset
# This gives us diversity in terms of movie era, genre, and popularity
SAMPLE_SIZE = 20
indices = np.linspace(0, len(moviesum_df) - 1, SAMPLE_SIZE, dtype=int)
filtered_df = moviesum_df.iloc[indices].copy()

# Calculate statistics
total_chars = filtered_df['script'].str.len().sum()
avg_chars = filtered_df['script'].str.len().mean()
median_chars = filtered_df['script'].str.len().median()

print(f"\nSelected {len(filtered_df)} movies for processing (evenly sampled across dataset)")
print(f"Total characters: {total_chars:,}")
print(f"Average characters per movie: {avg_chars:,.0f}")
print(f"Median characters per movie: {median_chars:,.0f}")

print("\nFirst 5 movies in sample:")
for idx, row in filtered_df.head(5).iterrows():
    script_length = len(row['script']) if pd.notna(row['script']) else 0
    print(f"  - {row['movie_name']} ({row['imdb_id']}) - {script_length:,} characters")

Loading MovieSum dataset...


Repo card metadata block was not found. Setting CardData to empty.


Total movies available: 1800

Selected 20 movies for processing (evenly sampled across dataset)
Total characters: 4,199,396
Average characters per movie: 209,970
Median characters per movie: 195,977

First 5 movies in sample:
  - 8MM_1999 (tt0134273) - 277,877 characters
  - White Christmas_1954 (tt0047673) - 149,853 characters
  - The Maltese Falcon_1941 (tt0033870) - 288,923 characters
  - Pariah_2011 (tt1233334) - 189,091 characters
  - Broadcast News_1987 (tt0092699) - 262,066 characters


In [14]:
"""
COST ESTIMATION METHODOLOGY
============================

GraphRAG processes documents through multiple stages, each using OpenAI APIs:

1. ENTITY EXTRACTION: Each text chunk is analyzed by LLM to extract entities and relationships
   - Input: Prompt (~1500 tokens) + Chunk (1200 tokens) = 2700 tokens per chunk
   - Output: Extracted entities/relationships (~750 tokens per chunk)
   - With max_gleanings=1, this happens TWICE per chunk (initial + refinement)

2. ENTITY SUMMARIZATION: Duplicate entities are merged and summarized
   - Estimate: ~50 entities per movie
   - Input: ~500 tokens per entity summary
   - Output: ~200 tokens per entity summary

3. COMMUNITY DETECTION: Graph algorithm (no LLM cost)

4. COMMUNITY REPORTS: LLM generates summaries for each community
   - Estimate: ~20 communities per movie  
   - Input: ~3000 tokens per community (entity descriptions)
   - Output: ~1500 tokens per community report

5. EMBEDDINGS: Text chunks and entities are embedded
   - Items: All chunks + all entities
   - Tokens: ~200 tokens per item average

OpenAI Pricing (as of January 2025):
- GPT-4.1-mini: $0.40 per 1M input tokens, $1.60 per 1M output tokens
- text-embedding-3-small: $0.020 per 1M tokens
"""

import tiktoken

# Configuration from notebook
CHUNK_SIZE = 1200  # Must match settings.yaml chunks.size
CHUNK_OVERLAP = 100
MAX_GLEANINGS = 1  # Means 2 passes total (initial + 1 refinement)

# Sample statistics
sample_size = len(filtered_df)
sample_total_chars = filtered_df['script'].str.len().sum()
sample_avg_chars = sample_total_chars / sample_size

# Full dataset statistics
full_dataset_size = len(moviesum_df)
full_dataset_chars = moviesum_df['script'].str.len().sum()
full_avg_chars = full_dataset_chars / full_dataset_size

def estimate_costs(num_movies, total_chars, avg_chars_per_movie):
    """Estimate GraphRAG processing costs"""
    
    # Convert characters to tokens (rough approximation: 4 chars per token)
    total_tokens = total_chars / 4
    avg_tokens_per_movie = avg_chars_per_movie / 4
    
    # Calculate chunks
    effective_chunk_step = CHUNK_SIZE - CHUNK_OVERLAP
    chunks_per_movie = avg_tokens_per_movie / effective_chunk_step
    total_chunks = num_movies * chunks_per_movie
    
    # Stage 1: Entity Extraction (with max_gleanings=1, runs 2x per chunk)
    passes = MAX_GLEANINGS + 1
    entity_input_per_chunk = (1500 + CHUNK_SIZE)  # prompt + chunk
    entity_output_per_chunk = 750
    entity_extraction_input = total_chunks * entity_input_per_chunk * passes
    entity_extraction_output = total_chunks * entity_output_per_chunk * passes
    
    # Stage 2: Entity Summarization
    entities_per_movie = 50
    total_entities = num_movies * entities_per_movie
    summarize_input = total_entities * 500
    summarize_output = total_entities * 200
    
    # Stage 3: Community Reports
    communities_per_movie = 20
    total_communities = num_movies * communities_per_movie
    community_input = total_communities * 3000
    community_output = total_communities * 1500
    
    # Stage 4: Embeddings
    embedding_items = total_chunks + total_entities
    embedding_tokens = embedding_items * 200
    
    # Total tokens
    total_llm_input = entity_extraction_input + summarize_input + community_input
    total_llm_output = entity_extraction_output + summarize_output + community_output
    
    # OpenAI Pricing (January 2025)
    gpt41_mini_input_price = 0.40 / 1_000_000   # GPT-4.1-mini
    gpt41_mini_output_price = 1.60 / 1_000_000  # GPT-4.1-mini
    embedding_price = 0.020 / 1_000_000
    
    # Calculate costs
    llm_input_cost = total_llm_input * gpt41_mini_input_price
    llm_output_cost = total_llm_output * gpt41_mini_output_price
    embedding_cost = embedding_tokens * embedding_price
    total_cost = llm_input_cost + llm_output_cost + embedding_cost
    
    # Time estimation (10 concurrent requests, ~100 tokens/sec output)
    entity_time = (entity_output_per_chunk / 100 + 0.5) * total_chunks * passes / 10 / 60
    summarize_time = (200 / 100 + 0.5) * total_entities / 10 / 60
    community_time = (1500 / 100 + 0.5) * total_communities / 10 / 60
    embedding_time = embedding_items / 1000 / 60
    total_time_minutes = entity_time + summarize_time + community_time + embedding_time + 1
    
    return {
        'num_movies': num_movies,
        'total_chars': total_chars,
        'total_chunks': int(total_chunks),
        'total_entities': int(total_entities),
        'total_communities': int(total_communities),
        'llm_input_tokens': int(total_llm_input),
        'llm_output_tokens': int(total_llm_output),
        'embedding_tokens': int(embedding_tokens),
        'llm_input_cost': llm_input_cost,
        'llm_output_cost': llm_output_cost,
        'embedding_cost': embedding_cost,
        'total_cost': total_cost,
        'total_time_hours': total_time_minutes / 60,
        'total_time_minutes': total_time_minutes
    }

# Calculate for sample
sample_est = estimate_costs(sample_size, sample_total_chars, sample_avg_chars)

# Calculate for full dataset
full_est = estimate_costs(full_dataset_size, full_dataset_chars, full_avg_chars)

print("=" * 80)
print("COST & TIME ESTIMATION FOR GRAPHRAG PROCESSING")
print("=" * 80)
print()

print(f"SAMPLE (20 movies):")
print(f"  Movies:                {sample_est['num_movies']:>10,}")
print(f"  Total characters:      {sample_est['total_chars']:>10,}")
print(f"  Text chunks:           {sample_est['total_chunks']:>10,}")
print(f"  Entities (estimated):  {sample_est['total_entities']:>10,}")
print(f"  Communities (est.):    {sample_est['total_communities']:>10,}")
print()
print(f"  LLM input tokens:      {sample_est['llm_input_tokens']:>10,}")
print(f"  LLM output tokens:     {sample_est['llm_output_tokens']:>10,}")
print(f"  Embedding tokens:      {sample_est['embedding_tokens']:>10,}")
print()
print(f"  💰 LLM input cost:      ${sample_est['llm_input_cost']:>9.2f}")
print(f"  💰 LLM output cost:     ${sample_est['llm_output_cost']:>9.2f}")
print(f"  💰 Embedding cost:      ${sample_est['embedding_cost']:>9.2f}")
print(f"  💰 TOTAL COST:          ${sample_est['total_cost']:>9.2f}")
print()
print(f"  ⏱️  Estimated time:      {sample_est['total_time_hours']:>9.1f} hours ({sample_est['total_time_minutes']:.0f} min)")
print()

print("=" * 80)
print(f"FULL DATASET (1,800 movies):")
print(f"  Movies:                {full_est['num_movies']:>10,}")
print(f"  Total characters:      {full_est['total_chars']:>10,}")
print(f"  Text chunks:           {full_est['total_chunks']:>10,}")
print(f"  Entities (estimated):  {full_est['total_entities']:>10,}")
print(f"  Communities (est.):    {full_est['total_communities']:>10,}")
print()
print(f"  LLM input tokens:      {full_est['llm_input_tokens']:>10,}")
print(f"  LLM output tokens:     {full_est['llm_output_tokens']:>10,}")
print(f"  Embedding tokens:      {full_est['embedding_tokens']:>10,}")
print()
print(f"  💰 LLM input cost:      ${full_est['llm_input_cost']:>9.2f}")
print(f"  💰 LLM output cost:     ${full_est['llm_output_cost']:>9.2f}")
print(f"  💰 Embedding cost:      ${full_est['embedding_cost']:>9.2f}")
print(f"  💰 TOTAL COST:          ${full_est['total_cost']:>9.2f}")
print()
print(f"  ⏱️  Estimated time:      {full_est['total_time_hours']:>9.1f} hours ({full_est['total_time_minutes']:.0f} min)")
print()

print("=" * 80)
print("SCALE COMPARISON:")
scale = full_dataset_size / sample_size
print(f"  Dataset scale:  {scale:.0f}x more movies")
print(f"  Cost scale:     {full_est['total_cost'] / sample_est['total_cost']:.1f}x more expensive")
print(f"  Time scale:     {full_est['total_time_minutes'] / sample_est['total_time_minutes']:.1f}x longer")
print()
print("Note: These are estimates. Actual costs may vary based on:")
print("  - Actual number of entities/relationships found per movie")
print("  - Community structure complexity")
print("  - API rate limits and retry behavior")
print("=" * 80)

COST & TIME ESTIMATION FOR GRAPHRAG PROCESSING

SAMPLE (20 movies):
  Movies:                        20
  Total characters:       4,199,396
  Text chunks:                  954
  Entities (estimated):       1,000
  Communities (est.):           400

  LLM input tokens:       6,853,804
  LLM output tokens:      2,231,612
  Embedding tokens:         390,881

  💰 LLM input cost:      $     2.74
  💰 LLM output cost:     $     3.57
  💰 Embedding cost:      $     0.01
  💰 TOTAL COST:          $     6.32

  ⏱️  Estimated time:            0.7 hours (41 min)

FULL DATASET (1,800 movies):
  Movies:                     1,800
  Total characters:      372,490,280
  Text chunks:               84,656
  Entities (estimated):      90,000
  Communities (est.):        36,000

  LLM input tokens:      610,147,161
  LLM output tokens:     198,985,322
  Embedding tokens:      34,931,376

  💰 LLM input cost:      $   244.06
  💰 LLM output cost:     $   318.38
  💰 Embedding cost:      $     0.70
  💰 TOTAL COST

## Step 4: Prepare Input Documents

GraphRAG expects text files in the input directory. We'll create one file per movie script.

In [15]:
import yaml

# GraphRAG configuration optimized for movie scripts
# Note: GraphRAG now uses a 'models' section with specific model IDs
settings = {
    "models": {
        "default_chat_model": {
            "api_key": "$OPENAI_API_KEY",
            "type": "openai_chat",
            "model": LLM_MODEL,
            "max_tokens": 4000,
            "temperature": 0.0,
            "top_p": 1.0,
        },
        "default_embedding_model": {
            "api_key": "$OPENAI_API_KEY",
            "type": "openai_embedding",
            "model": EMBEDDING_MODEL,
        },
    },
    "chunks": {
        "size": 1200,
        "overlap": 100,
        "group_by_columns": ["id"],
    },
    "input": {
        "type": "file",
        "file_type": "text",
        "base_dir": str(INPUT_DIR),
        "file_pattern": ".*\\\\.txt$$",  # Escaped for Python Template syntax
    },
    "cache": {
        "type": "file",
        "base_dir": str(GRAPHRAG_DIR / "cache"),
    },
    "storage": {
        "type": "file",
        "base_dir": str(OUTPUT_DIR),
    },
    "reporting": {
        "type": "file",
        "base_dir": str(GRAPHRAG_DIR / "reports"),
    },
    "extract_graph": {
        # Don't specify custom prompts - let GraphRAG use its built-in defaults
        "entity_types": [
            "character",
            "person",
            "location",
            "organization",
            "event",
            "object"
        ],
        "max_gleanings": 1,
    },
    "summarize_descriptions": {
        # Don't specify custom prompts - let GraphRAG use its built-in defaults
        "max_length": 500,
    },
    "community_reports": {
        # Don't specify custom prompts - let GraphRAG use its built-in defaults
        "max_length": 1500,
        "max_input_length": 8000,
    },
}

# Write settings.yaml
settings_path = GRAPHRAG_DIR / "settings.yaml"
with open(settings_path, 'w') as f:
    yaml.dump(settings, f, default_flow_style=False, sort_keys=False)

print(f"Configuration written to {settings_path}")
print("\nKey settings:")
print(f"  - LLM Model: {LLM_MODEL}")
print(f"  - Embedding Model: {EMBEDDING_MODEL}")
print(f"  - Chunk Size: {settings['chunks']['size']}")
print(f"  - API Key: Uses OPENAI_API_KEY from environment")

Configuration written to graphrag_workspace/settings.yaml

Key settings:
  - LLM Model: gpt-4.1-mini
  - Embedding Model: text-embedding-3-small
  - Chunk Size: 1200
  - API Key: Uses OPENAI_API_KEY from environment


## Step 5: Run GraphRAG Indexing

⚠️ **Warning**: This step will use OpenAI API credits.

Based on the cost estimation above:
- **Estimated cost for 20 movies**: ~$4.12
- **Estimated processing time**: ~0.5 hours (28 min)

For the full dataset (1,800 movies):
- Estimated cost: ~$370
- Estimated time: ~45 hours

In [ ]:
# Prepare input documents DataFrame for GraphRAG
# GraphRAG expects a DataFrame with specific columns:
# - id: document identifier
# - title: document title  
# - text: main content
# - creation_date: timestamp (optional but expected by some workflows)
from datetime import datetime

print("Preparing input documents for GraphRAG...")

input_docs = []
current_date = datetime.now()

for idx, row in filtered_df.iterrows():
    input_docs.append({
        'id': row['imdb_id'],
        'title': row['movie_name'],
        'text': row['script'],
        'creation_date': current_date  # Add creation date for all documents
    })

input_df = pd.DataFrame(input_docs)
print(f"Prepared {len(input_df)} documents for indexing")
print(f"\nSample document:")
print(f"  ID: {input_df.iloc[0]['id']}")
print(f"  Title: {input_df.iloc[0]['title']}")
print(f"  Text length: {len(input_df.iloc[0]['text']):,} characters")
print(f"  Creation date: {input_df.iloc[0]['creation_date']}")

# Run GraphRAG indexing using the programmatic API
from graphrag.api import build_index
from graphrag.config.load_config import load_config

print("\n" + "="*60)
print("Starting GraphRAG indexing...")
print("This may take several minutes depending on the amount of text.")
print("="*60 + "\n")

# Load configuration - use Path object, not string
config = load_config(root_dir=GRAPHRAG_DIR)

# Run indexing with the input documents
results = await build_index(
    config=config,
    input_documents=input_df,
    verbose=True
)

print("\n" + "="*60)
print("Indexing complete!")
print("="*60)

In [19]:
# Check results
print(f"\nCompleted {len(results)} workflow(s)")
for result in results:
    if result.errors:
        print(f"  ❌ Workflow '{result.workflow}' had {len(result.errors)} error(s)")
        for error in result.errors[:3]:  # Show first 3 errors
            print(f"     - {error}")
    else:
        print(f"  ✅ Workflow '{result.workflow}' completed successfully")

# Check output files (GraphRAG stores parquet files directly in output dir)
if OUTPUT_DIR.exists():
    parquet_files = list(OUTPUT_DIR.glob("*.parquet"))
    if parquet_files:
        print(f"\n✅ Generated {len(parquet_files)} output files:")
        for file in sorted(parquet_files):
            size_kb = file.stat().st_size / 1024
            print(f"  - {file.name} ({size_kb:.1f} KB)")
    else:
        print(f"\n⚠️  Warning: No parquet files found in {OUTPUT_DIR}")
else:
    print(f"\n⚠️  Warning: Output directory {OUTPUT_DIR} does not exist.")


Completed 9 workflow(s)
  ✅ Workflow 'create_base_text_units' completed successfully
  ✅ Workflow 'create_final_documents' completed successfully
  ✅ Workflow 'extract_graph' completed successfully
  ✅ Workflow 'finalize_graph' completed successfully
  ✅ Workflow 'extract_covariates' completed successfully
  ✅ Workflow 'create_communities' completed successfully
  ✅ Workflow 'create_final_text_units' completed successfully
  ✅ Workflow 'create_community_reports' completed successfully
  ✅ Workflow 'generate_text_embeddings' completed successfully

✅ Generated 6 output files:
  - communities.parquet (964.7 KB)
  - community_reports.parquet (9822.6 KB)
  - documents.parquet (1851.3 KB)
  - entities.parquet (1758.2 KB)
  - relationships.parquet (2080.7 KB)
  - text_units.parquet (2700.5 KB)
